# Day 1 | Lab 1.2: Robustness Patterns (Retries, Backoff, Fallback Providers)

**Duration:** ~1.5 hours

**Scenario.** Mixed-domain operational hardening (healthcare triage, e-commerce reviews, insurance batch claims) — preserved from the source notebook. The lab focuses on production reliability patterns, not domain.

**Learning Objectives.** By the end of this lab, you will be able to:
1. Identify common LLM-API errors (rate limits, timeouts, server errors) and classify them as retryable vs non-retryable.
2. Implement **retry logic with exponential backoff + jitter** to avoid thundering-herd problems.
3. Build a **1-level provider fallback** (OpenAI → Groq) for high-availability service.
4. Combine retries + fallback into a single production-ready robust caller function.
5. Add structured logging and reliability metrics to monitor LLM API health in production.

## *Architecture*

<svg width="100%" viewBox="0 0 680 430" xmlns="http://www.w3.org/2000/svg">
  <defs>
    <marker id="arrow" viewBox="0 0 10 10" refX="8" refY="5" markerWidth="6" markerHeight="6" orient="auto-start-reverse">
      <path d="M2 1L8 5L2 9" fill="none" stroke="#5F5E5A" stroke-width="1.5" stroke-linecap="round" stroke-linejoin="round"/>
    </marker>
  </defs>

  <rect x="120" y="30" width="160" height="40" rx="8" fill="#F1EFE8" stroke="#5F5E5A" stroke-width="0.5"/>
  <text x="200" y="50" text-anchor="middle" dominant-baseline="central" font-family="Helvetica, Arial, sans-serif" font-size="14" font-weight="500" fill="#2C2C2A">User request</text>

  <line x1="200" y1="70" x2="200" y2="108" stroke="#5F5E5A" stroke-width="1.5" marker-end="url(#arrow)"/>

  <rect x="60" y="110" width="280" height="64" rx="8" fill="#E6F1FB" stroke="#185FA5" stroke-width="0.5"/>
  <text x="200" y="132" text-anchor="middle" dominant-baseline="central" font-family="Helvetica, Arial, sans-serif" font-size="14" font-weight="500" fill="#042C53">Primary: OpenAI</text>
  <text x="200" y="152" text-anchor="middle" dominant-baseline="central" font-family="Helvetica, Arial, sans-serif" font-size="12" fill="#185FA5">gpt-4.1-mini</text>

  <line x1="340" y1="142" x2="380" y2="142" stroke="#888780" stroke-width="0.5" stroke-dasharray="3 3"/>
  <text x="386" y="136" dominant-baseline="central" font-family="Helvetica, Arial, sans-serif" font-size="12" fill="#444441">Retry up to 3×</text>
  <text x="386" y="152" dominant-baseline="central" font-family="Helvetica, Arial, sans-serif" font-size="12" fill="#444441">with exponential backoff + jitter</text>

  <line x1="200" y1="174" x2="200" y2="228" stroke="#5F5E5A" stroke-width="1.5" marker-end="url(#arrow)"/>
  <text x="215" y="201" dominant-baseline="central" font-family="Helvetica, Arial, sans-serif" font-size="12" fill="#444441">All retries failed?</text>

  <rect x="60" y="230" width="280" height="64" rx="8" fill="#FAEEDA" stroke="#854F0B" stroke-width="0.5"/>
  <text x="200" y="252" text-anchor="middle" dominant-baseline="central" font-family="Helvetica, Arial, sans-serif" font-size="14" font-weight="500" fill="#412402">Fallback: Groq</text>
  <text x="200" y="272" text-anchor="middle" dominant-baseline="central" font-family="Helvetica, Arial, sans-serif" font-size="12" fill="#854F0B">llama-3.3-70b</text>

  <line x1="340" y1="262" x2="380" y2="262" stroke="#888780" stroke-width="0.5" stroke-dasharray="3 3"/>
  <text x="386" y="256" dominant-baseline="central" font-family="Helvetica, Arial, sans-serif" font-size="12" fill="#444441">Single attempt</text>
  <text x="386" y="272" dominant-baseline="central" font-family="Helvetica, Arial, sans-serif" font-size="12" fill="#444441">(already a backup)</text>

  <line x1="200" y1="294" x2="200" y2="348" stroke="#5F5E5A" stroke-width="1.5" marker-end="url(#arrow)"/>
  <text x="215" y="321" dominant-baseline="central" font-family="Helvetica, Arial, sans-serif" font-size="12" fill="#444441">Also failed?</text>

  <rect x="80" y="350" width="240" height="44" rx="8" fill="#FCEBEB" stroke="#A32D2D" stroke-width="0.5"/>
  <text x="200" y="372" text-anchor="middle" dominant-baseline="central" font-family="Helvetica, Arial, sans-serif" font-size="14" font-weight="500" fill="#501313">Graceful error response</text>
</svg>

**Models Used**

| Role | Provider | Model | Why |
|---|---|---|---|
| Primary | OpenAI | `gpt-4.1-mini` | High quality, supports temperature |
| Fallback | Groq Cloud | `llama-3.3-70b-versatile` | Ultra-fast inference on LPU hardware |

---


## 1. Install Dependencies

In [ ]:
# Required packages for this lab — already installed in your local venv.
# To install standalone, uncomment the line(s) below:
# !pip install openai groq -q


## 2. API Key Configuration

In [ ]:
import os
env_path = r"C:\Users\prash\OneDrive\2026\Training\eClerx-GenAI-Training\.env"

# Local-venv pattern: load from .env if python-dotenv is available, otherwise rely on
# environment variables already set in your shell or venv activation script.
try:
    from dotenv import load_dotenv
    load_dotenv(env_path)
except ImportError:
    pass

# Verify keys are loaded (prints status, never prints actual values)
for key in ['OPENAI_API_KEY', 'GROQ_API_KEY']:
    status = '✅ Loaded' if os.environ.get(key) else '❌ MISSING'
    print(f'{key}: {status}')


OPENAI_API_KEY: ✅ Loaded
GROQ_API_KEY: ✅ Loaded


## 3. Initialize Clients & Imports

In [4]:
import time
import random
import logging
from datetime import datetime
from openai import OpenAI
from groq import Groq

# Import specific exception classes for targeted error handling
import openai

openai_client = OpenAI()
groq_client = Groq()

print("Clients initialized ✅")

Clients initialized ✅


---
## 4. Task 1 — Understanding Common API Errors

### Why This Matters
Before building retry logic, you need to know **what can go wrong**. LLM APIs fail in predictable ways — knowing the error types helps you write targeted handlers.

| Error Type | HTTP Status | Cause | Should Retry? |
|-----------|-------------|-------|---------------|
| **Rate Limit** | 429 | Too many requests per minute | ✅ Yes — wait and retry |
| **Server Error** | 500, 502, 503 | Provider infrastructure issue | ✅ Yes — transient |
| **Timeout** | 408 / SDK timeout | Slow response or network issue | ✅ Yes — may succeed on retry |
| **Authentication** | 401, 403 | Invalid/expired API key | ❌ No — fix the key |
| **Bad Request** | 400 | Malformed input, exceeded context | ❌ No — fix the request |
| **Content Filter** | 400 | Input/output blocked by safety filter | ❌ No — rephrase the request |

Let's trigger and catch these errors to understand the SDK's exception classes.

In [5]:
# --- Demonstrate error handling with OpenAI ---
# Test 1: Invalid API key (AuthenticationError)
print("Test 1: Invalid API key")
print("─" * 40)
try:
    bad_client = OpenAI(api_key="sk-invalid-key-12345")
    bad_client.responses.create(
        model="gpt-4.1-mini",
        input="Hello",
    )
except openai.AuthenticationError as e:
    print(f"✅ Caught AuthenticationError: {e.status_code}")
    print(f"   → This is NOT retryable. Fix the API key.")
    # Fixed: Access the error details directly from the exception object

    # code = getattr(e, "status_code", None) or getattr(e, "status", None)
    # mess = getattr(e, "message", None)
    # print(f"   → Details: {mess}")
except Exception as e: #any other error
    print(f"Caught: {type(e).__name__}: {e}")

Test 1: Invalid API key
────────────────────────────────────────
✅ Caught AuthenticationError: 401
   → This is NOT retryable. Fix the API key.


In [ ]:
# Test 2: Invalid model name (NotFoundError)
print("Test 2: Invalid model name")
print("─" * 40)
try:
    openai_client.responses.create(
        model="gpt-nonexistent-model",
        input="Hello",
    )
except openai.NotFoundError as e:
    print(f"✅ Caught NotFoundError: {e.status_code}")
    print(f"   → This is NOT retryable. Check model name.")
except Exception as e:
    print(f"Caught: {type(e).__name__}: {e}")

Test 2: Invalid model name
────────────────────────────────────────
Caught: BadRequestError: Error code: 400 - {'error': {'message': "The requested model 'gpt-nonexistent-model' does not exist.", 'type': 'invalid_request_error', 'param': 'model', 'code': 'model_not_found'}}


In [ ]:
# Test 3: Successful call (baseline for comparison)
print("Test 3: Successful call (baseline)")
print("─" * 40)
try:
    response = openai_client.responses.create(
        model="gpt-4.1-mini",
        input="Say 'API is working' in exactly 3 words.",
        max_output_tokens=20,
    )
    print(f"✅ Success: {response.output_text}")
except Exception as e:
    print(f"❌ Unexpected error: {type(e).__name__}: {e}")

Test 3: Successful call (baseline)
────────────────────────────────────────
✅ Success: API is working


### 📝 Key Takeaway
The critical design decision is: **which errors are retryable?** Rate limits (429) and server errors (5xx) are transient — retrying after a delay often succeeds. Authentication and bad request errors are permanent — retrying wastes time and money.

---
## 5. Task 2 — Retry with Exponential Backoff

### Business Scenario: Insurance Claims Processing Pipeline
An insurance company processes 500+ claims per hour through an LLM-powered summarization pipeline. During peak hours, rate limits are frequently hit. The pipeline needs to **automatically retry** without losing claims or overloading the API.

### What is Exponential Backoff?
Instead of retrying immediately (which floods the API harder), we wait progressively longer between retries:
- Retry 1: wait ~1 second
- Retry 2: wait ~2 seconds
- Retry 3: wait ~4 seconds

We also add **jitter** (small random offset) so that multiple clients hitting the same rate limit don't all retry at the exact same time.

In [ ]:
# --- Define retryable vs. non-retryable error categories ---
RETRYABLE_ERRORS = (
    openai.RateLimitError,      # 429 — too many requests
    openai.APITimeoutError,     # request timed out
    openai.InternalServerError, # 500 — server-side issue
    openai.APIConnectionError,  # network connectivity problem
)

NON_RETRYABLE_ERRORS = (
    openai.AuthenticationError, # 401/403 — bad API key
    openai.BadRequestError,     # 400 — invalid input
    openai.NotFoundError,       # 404 — wrong model name
)


def call_openai_with_retry(system_prompt, user_prompt, model="gpt-4.1-mini",
                            temperature=0.7, max_tokens=500,
                            max_retries=3, base_delay=1.0):
    """
    Call OpenAI Responses API with exponential backoff retry logic.

    Retries on: rate limit, timeout, server error, connection error.
    Fails fast on: auth error, bad request, not found.

    Backoff formula: delay = base_delay * (2 ^ attempt) + random jitter
    """
    for attempt in range(max_retries + 1):  # attempt 0 = first try
        try:
            start = time.time()
            response = openai_client.responses.create(
                model=model,
                instructions=system_prompt,
                input=user_prompt,
                temperature=temperature,
                max_output_tokens=max_tokens,
            )
            latency = round(time.time() - start, 2)

            return {
                "provider": "OpenAI",
                "model": model,
                "response": response.output_text,
                "latency_sec": latency,
                "attempts": attempt + 1,
                "status": "success",
            }

        except NON_RETRYABLE_ERRORS as e:
            # Permanent error — fail immediately, don't waste retries
            print(f"  ❌ Non-retryable error: {type(e).__name__}")
            raise

        except RETRYABLE_ERRORS as e:
            if attempt < max_retries:
                delay = base_delay * (2 ** attempt) + random.uniform(0, 0.5)
                print(f"  ⚠️ Attempt {attempt + 1} failed: {type(e).__name__}")
                print(f"     Retrying in {delay:.1f}s... ({max_retries - attempt} retries left)")
                time.sleep(delay)
            else:
                print(f"  ❌ All {max_retries + 1} attempts exhausted: {type(e).__name__}")
                raise

        except Exception as e:
            print(f"  ❌ Unexpected error: {type(e).__name__}: {e}")
            raise


print("Retry function defined ✅")

Retry function defined ✅


In [ ]:
# --- Test retry function with a valid request ---
# Business scenario: Auto-summarize a health insurance claim
system = "You are an insurance claims analyst. Summarize claims concisely."
claim = """Claim #HC-2025-4412: Policyholder Meera Shah, age 45, was admitted for
emergency appendectomy at Apollo Hospital, Mumbai on Feb 3, 2025. Total bill: ₹3.2L.
Policy: Star Health Comprehensive, Sum Insured ₹10L, no sub-limits on surgery.
Pre-authorization obtained. Discharged on Feb 5. All documents submitted."""

print("Testing retry function with a valid request...")
result = call_openai_with_retry(system, claim, max_tokens=200)

print(f"\n✅ Success in {result['attempts']} attempt(s) | {result['latency_sec']}s")
print(f"\n{result['response']}")

Testing retry function with a valid request...

✅ Success in 1 attempt(s) | 2.87s

Claim Summary for HC-2025-4412:
Policyholder: Meera Shah, 45 years old
Hospital: Apollo Hospital, Mumbai
Admission Date: Feb 3, 2025
Procedure: Emergency appendectomy
Total Bill: ₹3.2 Lakhs
Policy: Star Health Comprehensive, Sum Insured ₹10 Lakhs, no surgery sub-limits
Pre-authorization: Obtained
Discharge Date: Feb 5, 2025
Status: All documents submitted for processing


In [ ]:
# --- Demonstrate that non-retryable errors fail immediately ---
print("Testing: Non-retryable error should fail immediately (no wasted retries)...")
print("─" * 60)

real_client = openai_client  # save real client
try:
    # Temporarily inject a broken client to simulate auth failure
    globals()['openai_client'] = OpenAI(api_key="sk-broken-key")
    call_openai_with_retry("test", "test", max_retries=3)
except openai.AuthenticationError:
    print("\n✅ Correctly failed immediately — no retries wasted on a permanent error")
except Exception as e:
    print(f"\nCaught: {type(e).__name__}")
finally:
    globals()['openai_client'] = real_client  # restore real client
    print("Real client restored ✅")

Testing: Non-retryable error should fail immediately (no wasted retries)...
────────────────────────────────────────────────────────────
  ❌ Non-retryable error: AuthenticationError

✅ Correctly failed immediately — no retries wasted on a permanent error
Real client restored ✅


### 📝 Discussion Points
- **Exponential backoff** prevents the "thundering herd" problem where all clients retry at the same instant
- **Jitter** adds randomness so retries from different pipeline workers don't collide
- **Non-retryable error detection** saves time and cost — no point retrying a bad API key 3 times

---
## 6. Task 3 — Fallback Provider Pattern (OpenAI → Groq)

### Business Scenario: Real-Time Banking Chatbot
A retail bank's customer service chatbot handles 10,000+ conversations daily. If OpenAI's API goes down or is rate-limited, the chatbot must **not** go offline. Instead, it silently switches to Groq (Llama 3.3) so customers never notice the disruption.

This is the **1-level fallback pattern**: Primary → Fallback → Graceful error.

In [ ]:
def call_groq_fallback(system_prompt, user_prompt, temperature=0.7, max_tokens=500):
    """Fallback provider: Groq Cloud running Llama 3.3 70B."""
    start = time.time()
    response = groq_client.chat.completions.create(
        model="llama-3.3-70b-versatile",
        messages=[
            {"role": "system", "content": system_prompt},
            {"role": "user", "content": user_prompt}
        ],
        temperature=temperature,
        max_tokens=max_tokens,
    )
    latency = round(time.time() - start, 2)

    return {
        "provider": "Groq (Fallback)",
        "model": "llama-3.3-70b-versatile",
        "response": response.choices[0].message.content,
        "latency_sec": latency,
        "status": "success_via_fallback",
    }


def call_with_fallback(system_prompt, user_prompt, temperature=0.7, max_tokens=500):
    """
    Primary + Fallback pattern:
      1. Try OpenAI with up to 3 retries
      2. If all retries fail → try Groq once
      3. If Groq also fails → return graceful error dict

    Non-retryable errors (bad key, bad request) skip the fallback.
    """
    # Step 1: Try primary with retries
    try:
        print("🔵 Trying primary: OpenAI (gpt-4.1-mini)...")
        result = call_openai_with_retry(
            system_prompt, user_prompt,
            temperature=temperature, max_tokens=max_tokens,
            max_retries=3, base_delay=1.0
        )
        print(f"   ✅ Primary succeeded in {result['attempts']} attempt(s)")
        return result

    except NON_RETRYABLE_ERRORS as e:
        # Our code/config is broken — fallback won't help
        print(f"   ❌ Non-retryable — skipping fallback: {type(e).__name__}")
        return {
            "provider": "None", "model": "N/A",
            "response": f"Error: {type(e).__name__} — {str(e)[:200]}",
            "latency_sec": -1, "status": "permanent_error",
        }

    except Exception as primary_error:
        # Primary exhausted retries — switch to fallback
        print(f"\n🟡 Primary failed after retries. Switching to fallback: Groq (Llama 3.3)...")

        # Step 2: Try fallback
        try:
            result = call_groq_fallback(
                system_prompt, user_prompt,
                temperature=temperature, max_tokens=max_tokens
            )
            print(f"   ✅ Fallback succeeded | {result['latency_sec']}s")
            return result

        except Exception as fallback_error:
            print(f"   ❌ Fallback also failed: {type(fallback_error).__name__}")
            return {
                "provider": "None", "model": "N/A",
                "response": "Service temporarily unavailable. Please try again later.",
                "latency_sec": -1, "status": "all_providers_failed",
                "primary_error": str(primary_error)[:200],
                "fallback_error": str(fallback_error)[:200],
            }


print("Fallback functions defined ✅")

Fallback functions defined ✅


In [ ]:
# --- Test 1: Normal call (primary succeeds on first try) ---
print("═" * 60)
print("Test 1: Primary provider succeeds (happy path)")
print("═" * 60)

system = "You are a helpful retail banking assistant. Be concise."
query = "What documents do I need to open a salary account at your bank?"

result = call_with_fallback(system, query, max_tokens=200)
print(f"\nProvider used: {result['provider']}")
print(f"Status: {result['status']}")
print(f"\n{result['response']}")

════════════════════════════════════════════════════════════
Test 1: Primary provider succeeds (happy path)
════════════════════════════════════════════════════════════
🔵 Trying primary: OpenAI (gpt-4.1-mini)...
   ✅ Primary succeeded in 1 attempt(s)

Provider used: OpenAI
Status: success

To open a salary account, you typically need:

1. Proof of identity (e.g., passport, driver's license, Aadhaar card)
2. Proof of address (e.g., utility bill, rental agreement)
3. Employment proof or salary certificate from your employer
4. Recent passport-sized photographs
5. PAN card or Form 60/61 for tax purposes

Requirements may vary slightly, so please check with the specific branch or our website for detailed information.


In [ ]:
# --- Test 2: Force primary failure → observe fallback in action ---
print("═" * 60)
print("Test 2: Primary fails → Fallback (Groq) handles the request")
print("═" * 60)

# Temporarily break the OpenAI client to simulate an outage
real_client = openai_client
globals()['openai_client'] = OpenAI(api_key="sk-broken-for-testing")

system = "You are a helpful banking assistant. Be concise."
query = "How do I set up auto-pay for my credit card bill?"

result = call_with_fallback(system, query, max_tokens=200)
print(f"\nProvider used: {result['provider']}")
print(f"Status: {result['status']}")
print(f"\n{result['response']}")

# Restore the real client
globals()['openai_client'] = real_client
print("\nReal client restored ✅")

════════════════════════════════════════════════════════════
Test 2: Primary fails → Fallback (Groq) handles the request
════════════════════════════════════════════════════════════
🔵 Trying primary: OpenAI (gpt-4.1-mini)...
  ❌ Non-retryable error: AuthenticationError
   ❌ Non-retryable — skipping fallback: AuthenticationError

Provider used: None
Status: permanent_error

Error: AuthenticationError — Error code: 401 - {'error': {'message': 'Incorrect API key provided: sk-broke*********ting. You can find your API key at https://platform.openai.com/account/api-keys.', 'type': 'invalid_request_error'

Real client restored ✅


### 📝 Discussion Points
- The end-user never sees the provider switch — the response looks the same
- Groq + Llama provides a different quality profile but keeps the service alive
- In production, you'd log which provider served each request for quality monitoring

---
## 7. Task 4 — Production-Ready Robust Caller with Logging

### Business Scenario: E-Commerce Product Review Summarizer
An e-commerce platform summarizes thousands of customer reviews daily. The pipeline must be **bulletproof** — with retries, fallback, structured logging, and timing metadata for every call.

We combine everything into a single production-grade function.

In [ ]:
# --- Structured logging setup ---
logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s | %(levelname)-7s | %(message)s",
    datefmt="%H:%M:%S"
)
logger = logging.getLogger("llm_robustness")


def robust_llm_call(system_prompt, user_prompt, temperature=0.7,
                     max_tokens=500, max_retries=3, base_delay=1.0,
                     request_id=None):
    """
    Production-grade LLM caller combining:
      - Exponential backoff retries on transient errors
      - 1-level fallback (OpenAI → Groq)
      - Structured logging for monitoring
      - Request tracking via request_id

    Returns a dict with response, provider info, timing, and status.
    """
    req_id = request_id or f"req-{int(time.time())}"
    call_log = {
        "request_id": req_id,
        "start_time": datetime.now().isoformat(),
        "primary_attempts": 0,
        "used_fallback": False,
    }

    # --- Primary: OpenAI with retries ---
    for attempt in range(max_retries + 1):
        call_log["primary_attempts"] = attempt + 1
        try:
            logger.info(f"[{req_id}] Primary attempt {attempt + 1}/{max_retries + 1}")
            start = time.time()

            response = openai_client.responses.create(
                model="gpt-4.1-mini",
                instructions=system_prompt,
                input=user_prompt,
                temperature=temperature,
                max_output_tokens=max_tokens,
            )
            latency = round(time.time() - start, 2)
            logger.info(f"[{req_id}] ✅ Primary succeeded | {latency}s")

            call_log.update({
                "provider": "OpenAI", "model": "gpt-4.1-mini",
                "response": response.output_text,
                "latency_sec": latency, "status": "success",
                "input_tokens": response.usage.input_tokens,
                "output_tokens": response.usage.output_tokens,
            })
            return call_log

        except NON_RETRYABLE_ERRORS as e:
            logger.error(f"[{req_id}] Non-retryable: {type(e).__name__}")
            call_log.update({
                "provider": "None", "response": f"Error: {e}",
                "status": "permanent_error", "error_type": type(e).__name__,
            })
            return call_log

        except RETRYABLE_ERRORS as e:
            if attempt < max_retries:
                delay = base_delay * (2 ** attempt) + random.uniform(0, 0.5)
                logger.warning(f"[{req_id}] {type(e).__name__} — retry in {delay:.1f}s")
                time.sleep(delay)
            else:
                logger.error(f"[{req_id}] Primary exhausted after {max_retries + 1} attempts")

        except Exception as e:
            logger.error(f"[{req_id}] Unexpected: {type(e).__name__}: {str(e)[:100]}")
            if attempt == max_retries:
                break

    # --- Fallback: Groq ---
    call_log["used_fallback"] = True
    logger.info(f"[{req_id}] 🔄 Switching to fallback: Groq (Llama 3.3)")

    try:
        start = time.time()
        response = groq_client.chat.completions.create(
            model="llama-3.3-70b-versatile",
            messages=[
                {"role": "system", "content": system_prompt},
                {"role": "user", "content": user_prompt}
            ],
            temperature=temperature,
            max_tokens=max_tokens,
        )
        latency = round(time.time() - start, 2)
        logger.info(f"[{req_id}] ✅ Fallback succeeded | {latency}s")

        call_log.update({
            "provider": "Groq (Fallback)", "model": "llama-3.3-70b-versatile",
            "response": response.choices[0].message.content,
            "latency_sec": latency, "status": "success_via_fallback",
            "input_tokens": response.usage.prompt_tokens,
            "output_tokens": response.usage.completion_tokens,
        })
        return call_log

    except Exception as e:
        logger.critical(f"[{req_id}] ❌ Both providers failed!")
        call_log.update({
            "provider": "None", "model": "N/A",
            "response": "Service temporarily unavailable. Please try again later.",
            "latency_sec": -1, "status": "all_providers_failed",
            "error": str(e)[:200],
        })
        return call_log


print("Production robust caller defined ✅")

Production robust caller defined ✅


---
## 8. Task 5 — End-to-End Testing with Real Business Scenarios

Let's run the robust caller through multiple domain scenarios.

In [ ]:
# --- Scenario 1: Healthcare — Patient triage summary ---
print("═" * 60)
print("Scenario 1: Healthcare — Patient Triage Summary")
print("═" * 60)

result = robust_llm_call(
    system_prompt="You are an emergency department triage assistant. Summarize for the attending physician.",
    user_prompt="""Patient: Male, 55, chest pain radiating to left arm for 2 hours.
History: Type 2 diabetes (10 yrs), hypertension, ex-smoker (quit 2 yrs ago).
Vitals: BP 165/95, HR 102, SpO2 96%, afebrile. ECG: ST elevation in leads II, III, aVF.
Troponin pending. Aspirin 325mg and nitro sublingual given.""",
    max_tokens=200,
    request_id="triage-001"
)

print(f"\nProvider: {result['provider']} | Status: {result['status']}")
print(f"Attempts: {result['primary_attempts']} | Fallback used: {result['used_fallback']}")
print(f"\n{result['response']}")

════════════════════════════════════════════════════════════
Scenario 1: Healthcare — Patient Triage Summary
════════════════════════════════════════════════════════════

Provider: OpenAI | Status: success
Attempts: 1 | Fallback used: False

55-year-old male presenting with 2-hour history of chest pain radiating to left arm. Significant history includes type 2 diabetes (10 years), hypertension, and ex-smoker (quit 2 years ago). Vitals show elevated BP (165/95), tachycardia (HR 102), SpO2 96%, afebrile. ECG reveals ST elevation in leads II, III, aVF consistent with an acute inferior STEMI. Troponin pending. Initial management with aspirin 325 mg and sublingual nitroglycerin administered. Patient requires urgent cardiology evaluation for reperfusion therapy.


In [ ]:
# --- Scenario 2: E-Commerce — Product review summarization ---
print("═" * 60)
print("Scenario 2: E-Commerce — Product Review Summary")
print("═" * 60)

result = robust_llm_call(
    system_prompt="Summarize customer reviews into key pros, cons, and verdict in 2-3 sentences.",
    user_prompt="""Reviews for Samsung Galaxy S25 Ultra (500+ reviews aggregated):
- Camera is phenomenal, especially night mode and 200MP sensor
- Battery lasts full day with heavy use, 65% charge in 30 min
- S-Pen integration is seamless for note-taking
- Phone is heavy (233g) and slippery without a case
- Price ₹1.3L feels overpriced to many reviewers
- One UI has too many pre-installed Samsung apps""",
    max_tokens=150,
    request_id="review-042"
)

print(f"\nProvider: {result['provider']} | Status: {result['status']}")
print(f"\n{result['response']}")

════════════════════════════════════════════════════════════
Scenario 2: E-Commerce — Product Review Summary
════════════════════════════════════════════════════════════

Provider: OpenAI | Status: success

Key pros of the Samsung Galaxy S25 Ultra include its phenomenal camera performance with a standout 200MP sensor and night mode, long-lasting battery with fast charging, and seamless S-Pen integration for productivity. On the downside, the phone is quite heavy and slippery without a case, its price of ₹1.3L is considered steep by many, and One UI comes with an excess of pre-installed Samsung apps. Overall, it is a powerful flagship device ideal for photography and productivity but may feel overpriced and bulky for some users.


In [ ]:
# --- Scenario 3: Batch processing — 5 insurance claims ---
print("═" * 60)
print("Scenario 3: Batch Processing — Insurance Claim Assessments")
print("═" * 60)

claims = [
    "Claim #A101: Vehicle rear-ended at signal. Bumper + taillight damage. Estimate ₹45,000.",
    "Claim #A102: Water damage to apartment from upstairs pipe burst. Flooring + walls. Estimate ₹2.8L.",
    "Claim #A103: Laptop stolen from checked baggage on Delhi-Mumbai flight. ₹1.2L, 8 months old.",
    "Claim #A104: Dengue hospitalization, 5 days at Max Hospital. Bill ₹1.1L. Cashless claim.",
    "Claim #A105: Shop fire from electrical short. Inventory + fixtures destroyed. Estimate ₹12L.",
]

system = "You are an insurance claims processor. Give a 1-line assessment: LIKELY COVERED / UNLIKELY / NEEDS REVIEW — and the key factor."

batch_results = []
for i, claim in enumerate(claims):
    print(f"\nProcessing claim {i+1}/5...")
    result = robust_llm_call(system, claim, max_tokens=80, request_id=f"batch-{i+1}")
    batch_results.append(result)
    time.sleep(0.3)  # small delay to respect rate limits

# Print results summary
print("\n" + "═" * 60)
print("BATCH RESULTS SUMMARY")
print("═" * 60)
for i, r in enumerate(batch_results):
    claim_id = claims[i].split(":")[0]
    print(f"\n{claim_id} | {r['provider']} | {r['latency_sec']}s")
    print(f"  → {r['response'][:150]}")

════════════════════════════════════════════════════════════
Scenario 3: Batch Processing — Insurance Claim Assessments
════════════════════════════════════════════════════════════

Processing claim 1/5...

Processing claim 2/5...

Processing claim 3/5...

Processing claim 4/5...

Processing claim 5/5...

════════════════════════════════════════════════════════════
BATCH RESULTS SUMMARY
════════════════════════════════════════════════════════════

Claim #A101 | OpenAI | 0.73s
  → LIKELY COVERED – Rear-end collision with clear vehicle damage and reasonable repair estimate.

Claim #A102 | OpenAI | 0.86s
  → LIKELY COVERED — Water damage from a sudden pipe burst is typically covered under standard home insurance policies.

Claim #A103 | OpenAI | 0.7s
  → NEEDS REVIEW — Verify airline liability policy for checked baggage theft and policy coverage limits.

Claim #A104 | OpenAI | 1.39s
  → LIKELY COVERED — Dengue hospitalization is typically covered under inpatient hospitalization benefits.


---
## 9. Task 6 — Reliability Metrics Dashboard

### Business Scenario: Operations Monitoring
In production, you need visibility into how often calls succeed, fail, or use fallback. This data drives provider selection and SLA negotiations.

In [ ]:
def compute_reliability_metrics(results):
    """Compute reliability metrics from a batch of robust_llm_call results."""
    total = len(results)
    if total == 0:
        return {}

    success_primary = sum(1 for r in results if r["status"] == "success")
    success_fallback = sum(1 for r in results if r["status"] == "success_via_fallback")
    failed = sum(1 for r in results if r["status"] in ["permanent_error", "all_providers_failed"])

    latencies = [r["latency_sec"] for r in results if r["latency_sec"] > 0]
    avg_latency = round(sum(latencies) / len(latencies), 2) if latencies else 0

    avg_attempts = round(
        sum(r.get("primary_attempts", 1) for r in results) / total, 1
    )

    return {
        "total_requests": total,
        "success_primary": success_primary,
        "success_fallback": success_fallback,
        "success_rate": f"{(success_primary + success_fallback) / total * 100:.1f}%",
        "failed": failed,
        "avg_latency_sec": avg_latency,
        "avg_primary_attempts": avg_attempts,
        "fallback_usage": f"{success_fallback / total * 100:.1f}%",
    }


metrics = compute_reliability_metrics(batch_results)

print("📊 RELIABILITY METRICS")
print("═" * 40)
for key, value in metrics.items():
    print(f"  {key:25s}: {value}")

📊 RELIABILITY METRICS
════════════════════════════════════════
  total_requests           : 5
  success_primary          : 5
  success_fallback         : 0
  success_rate             : 100.0%
  failed                   : 0
  avg_latency_sec          : 0.89
  avg_primary_attempts     : 1.0
  fallback_usage           : 0.0%


### 📝 What to Monitor in Production

| Metric | What It Tells You | Action Threshold |
|--------|-------------------|------------------|
| **Success rate** | Overall system health | Alert if < 99% |
| **Fallback usage** | Primary provider reliability | Investigate if > 5% |
| **Avg retries per request** | Frequency of transient errors | Investigate if > 1.5 |
| **P95 latency** | Tail latency affecting UX | Alert if > 5s |
| **Error type distribution** | Root cause patterns | Review weekly |

---
## 10. Quick Reference — Error Handling Cheat Sheet

### OpenAI SDK Exception Classes
```python
import openai

# Retryable (transient) errors:
openai.RateLimitError        # 429 — too many requests
openai.APITimeoutError       # request timed out
openai.InternalServerError   # 500 — server-side issue
openai.APIConnectionError    # network problem

# Non-retryable (permanent) errors:
openai.AuthenticationError   # 401/403 — bad API key
openai.BadRequestError       # 400 — invalid input
openai.NotFoundError         # 404 — wrong model name
```

### Exponential Backoff Formula
```python
delay = base_delay * (2 ** attempt) + random.uniform(0, jitter)
# attempt=0: ~1.0-1.5s  |  attempt=1: ~2.0-2.5s  |  attempt=2: ~4.0-4.5s
```

### Fallback Pattern Summary
```python
try:
    result = primary_with_retry(...)   # OpenAI + retries
except RetryableError:
    result = fallback(...)             # Groq single attempt
except NonRetryableError:
    raise                              # Don't fallback on bad input
```

---
## 11. Conclusion & Key Takeaways

### What We Covered

| Task | Key Takeaway |
|---|---|
| **Error classification** | Distinguish retryable (429, 5xx, timeout) from non-retryable (401, 400) |
| **Exponential backoff** | Wait 1s → 2s → 4s between retries; add jitter to prevent collisions |
| **Fallback provider** | OpenAI → Groq (1-level) keeps your service alive during outages |
| **Production caller** | Combined retry + fallback + logging in a single reusable function |
| **Reliability metrics** | Track success rate, fallback usage, latency for ops visibility |

### Production Insights
- **Always classify errors before retrying** — retrying a bad API key 3 times wastes 7+ seconds for nothing.
- **Log every API call** with provider, latency, and status — this data is critical for debugging and cost tracking.
- **Set timeout limits** on LLM calls (e.g., 30 s) to prevent pipelines from hanging indefinitely.
- **Choose complementary fallback providers** — if OpenAI is primary, use a non-OpenAI fallback.

## 12. Stretch Exercise (Optional)
1. **2-level fallback chain** — extend the robust caller to OpenAI → Anthropic → Groq with the same retry policy on each.
2. **Circuit breaker** — after N consecutive failures on the primary, skip it for M minutes (e.g., `pybreaker` or a hand-rolled state machine).
3. **Per-provider timeout tuning** — measure P95 latency for each provider over 50 calls and set a request-level timeout at P95 × 1.5.
4. **Dead-letter queue** — for permanent failures, write the request payload + error to a JSON-lines file for later replay.
5. **Replace structured logging with `structlog`** and emit one JSON event per call with fields `request_id`, `provider`, `attempt`, `latency_ms`, `status`.

---

**Next Lab:** Lab 1.3 — Structured Outputs & Multi-Provider Comparison 📋
